# Chapter `2.2` - Agent State Management

### Importing the necessary libraries

In [ ]:
# Base utils.
import warnings
from dotenv import load_dotenv

from IPython.display import Markdown

# Model init. and invocation
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

# State management
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langchain.agents import AgentState

from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver

In [25]:
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Specific LangChain / LangGraph / Pydantic noise
warnings.filterwarnings("ignore", module="langchain")
warnings.filterwarnings("ignore", module="langgraph")
warnings.filterwarnings("ignore", module="pydantic")

load_dotenv()

# TODO: Environment variable config.

True

### Mutable state management

#### Providing a _mutable_ state

In [86]:
class ColourState(AgentState):
    favourite_colour: str = "blue"
    least_favourite_colour: str = "red"

#### Tools for allowing the agent to _access_ and _modify_ runtime state

In [59]:
@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user."""
    return runtime.state.get("favourite_colour", "blue")


@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user."""
    return runtime.state.get("least_favourite_colour", "blue")


@tool
def update_favourite_color(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they have revealed it to you."""
    return Command[tuple[()]](
        update={
            "favourite_colour": favourite_colour,
            "messages": [
                ToolMessage(
                    "I have successfully updated your favourite colour.",
                    tool_call_id=runtime.tool_call_id,
                ),
            ],
        }
    )

#### Agent initialization

In [ ]:
model = "ollama:gemma4:31b-cloud"

agent_updt = create_agent(
    model=model,
    state_schema=ColourState,
    tools=[get_favourite_colour, get_least_favourite_colour, update_favourite_color],
    checkpointer=InMemorySaver()
)

#### Now, testing across multiple calls

In [76]:
response = agent_updt.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    {"configurable": {"thread_id": "1"}},
)

Markdown(response["messages"][-1].content)

Your favourite colour is slate.

In [80]:
response = agent_updt.invoke(
    {"messages": [HumanMessage(content="Change my favourite colour to blue.")]},
    {"configurable": {"thread_id": "1"}},
)

Markdown(response["messages"][-1].content)

OK. I've changed your favourite colour to blue.

In [87]:
response = agent_updt.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    {"configurable": {"thread_id": "1"}},
)

Markdown(response["messages"][-1].content)

Your favourite colour is blue.